# 🐝 Heady Colab Runtime — All-in-One

**Multi-transport MCP bridge + GPU 3D vector space + HeadyMCP services.**

This notebook boots the full Heady infrastructure in Colab:
- 39 MCP tools via stdio, SSE, HTTP REST, and WebSocket
- GPU-accelerated 3D vector store (384D embeddings)
- Template HeadyBee/HeadySwarm auto-generation
- ngrok tunnel for remote IDE access

---
**© 2026 Heady Systems LLC. PROPRIETARY AND CONFIDENTIAL.**

## Cell 1: Bootstrap — Install Node.js + Dependencies

In [ ]:
%%bash
# ═══ Heady Bootstrap ═══
echo '🐝 Heady Colab Bootstrap'
echo '════════════════════════'

# Install Node.js 22
if ! command -v node &> /dev/null || [[ $(node -v) != v22* ]]; then
    curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -
    sudo apt-get install -y nodejs
fi
echo "Node: $(node -v)"
echo "NPM: $(npm -v)"

# Clone/update Heady repo
HEADY_DIR='/content/Heady'
if [ -d "$HEADY_DIR" ]; then
    cd $HEADY_DIR && git pull
else
    git clone https://github.com/HeadyMe/Heady-pre-production.git $HEADY_DIR
    cd $HEADY_DIR
fi

# Install npm deps
npm install --production 2>&1 | tail -3

# Install Python GPU packages
pip install -q sentence-transformers torch pyngrok

echo '✅ Bootstrap complete'

## Cell 2: GPU Detection + 3D Vector Space Init

In [ ]:
import torch
import json

gpu_info = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count() if torch.cuda.is_available() else 0,
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'memory_gb': round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1) if torch.cuda.is_available() else 0,
}

print(f'🐝 GPU Status')
print(f'═══════════════')
print(f'  CUDA: {gpu_info["cuda_available"]}')
print(f'  GPU:  {gpu_info["device_name"]}')
print(f'  VRAM: {gpu_info["memory_gb"]}GB')
print(f'  Devices: {gpu_info["device_count"]}')

# Store for Node.js
with open('/content/gpu_config.json', 'w') as f:
    json.dump(gpu_info, f)

print(f'\n✅ GPU config saved')

## Cell 3: Configure Secrets (ngrok, API keys)

In [ ]:
import os
from google.colab import userdata

# Load secrets from Colab Secrets
secrets = {}
for key in ['NGROK_TOKEN', 'HEADY_API_KEY', 'HEADY_BRAIN_URL', 'HEADY_MANAGER_URL']:
    try:
        secrets[key] = userdata.get(key)
        os.environ[key] = secrets[key]
        print(f'  ✅ {key}: loaded')
    except Exception:
        print(f'  ⚠ {key}: not set (optional)')

# Required: set transport to HTTP for Colab
os.environ['HEADY_MCP_TRANSPORT'] = 'http'
os.environ['HEADY_MCP_PORT'] = '8420'

print(f'\n✅ Secrets configured')

## Cell 4: Launch HeadyMCP Multi-Transport Bridge

In [ ]:
import subprocess, time, json, urllib.request

HEADY_DIR = '/content/Heady'

# Launch the MCP bridge in the background
proc = subprocess.Popen(
    ['node', 'src/mcp/colab-mcp-bridge.js'],
    cwd=HEADY_DIR,
    env={**dict(os.environ)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for boot
time.sleep(3)

# Read output
import select
while select.select([proc.stdout], [], [], 0.1)[0]:
    line = proc.stdout.readline().decode()
    if line: print(line, end='')

# Health check
try:
    req = urllib.request.Request('http://localhost:8420/health')
    with urllib.request.urlopen(req, timeout=5) as resp:
        health = json.loads(resp.read())
        print(f'\n✅ Bridge healthy: {health["service"]} v{health["version"]}')
        print(f'   Transports: {json.dumps(health["transports"])}')
        print(f'   Vectors: {health["vectorSpace"]["dimensions"]}D, GPU={health["vectorSpace"]["gpu"]}')
except Exception as e:
    print(f'❌ Health check failed: {e}')
    print(proc.stdout.read().decode())

## Cell 5: Verify All 39 MCP Tools

In [ ]:
import urllib.request, json

req = urllib.request.Request('http://localhost:8420/mcp/tools')
with urllib.request.urlopen(req) as resp:
    tools = json.loads(resp.read())

print(f'🐝 {tools["count"]} MCP Tools Available')
print(f'══════════════════════════════')
for t in tools['tools']:
    print(f'  mcp_Heady_{t["name"]}')

# Test vector store/search
import random
test_vec = [random.uniform(-1, 1) for _ in range(384)]

# Store
data = json.dumps({'embedding': test_vec, 'metadata': {'test': True, 'source': 'colab'}}).encode()
req = urllib.request.Request('http://localhost:8420/vector/store', data=data, headers={'Content-Type': 'application/json'})
with urllib.request.urlopen(req) as resp:
    store_result = json.loads(resp.read())

# Search
data = json.dumps({'embedding': test_vec, 'topK': 1}).encode()
req = urllib.request.Request('http://localhost:8420/vector/search', data=data, headers={'Content-Type': 'application/json'})
with urllib.request.urlopen(req) as resp:
    search_result = json.loads(resp.read())

print(f'\n✅ Vector Store: index={store_result["index"]}')
print(f'✅ Vector Search: score={search_result["results"][0]["score"]:.4f}')
print(f'\n🐝 All systems operational')

## Cell 6: Interactive — Call Any Heady Tool

In [ ]:
import urllib.request, json

def heady(tool_name, **kwargs):
    """Call any Heady MCP tool from Colab."""
    data = json.dumps({'name': tool_name, 'arguments': kwargs}).encode()
    req = urllib.request.Request(
        'http://localhost:8420/mcp/tools/call',
        data=data,
        headers={'Content-Type': 'application/json'}
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        result = json.loads(resp.read())
    # Parse the inner text content
    if 'content' in result:
        text = result['content'][0]['text']
        try:
            return json.loads(text)
        except:
            return text
    return result


# Examples:
# heady('heady_chat', message='Hello from Colab!')
# heady('heady_vector_stats')
# heady('heady_template_stats')
# heady('heady_health')
# heady('heady_deep_scan', directory='/content/Heady')

print('🐝 heady() function ready — call any of the 39 tools:')
print('   heady("heady_chat", message="Hello!")')
print('   heady("heady_vector_stats")')
print('   heady("heady_health")')